# Curation

In this tutorial we will show view and curate spike sorting results. We will first do some manual curation with the SpikeInterfac-GUI, then show some automated tools for unit labeling and merging.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import json

import spikeinterface as si
import spikeinterface.curation as sc
import spikeinterface.widgets as sw

import matplotlib.pyplot as plt

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
%matplotlib widget

In [ ]:
si.set_global_job_kwargs(n_jobs=0.5)

In [ ]:
base_data_folder = Path("/home/alessio/Documents/Data/SpikeInterface-Training-Data")
results_folder = Path("results/")

In [ ]:
analyzer_name = "full_analyzer_lupin_000939.zarr"
analyzer_folder = base_data_folder / analyzer_name

In [ ]:
analyzer = si.load(analyzer_folder, load_extensions=True)
analyzer

## Manual curation

We can use the SpikeInterface GUI for manual spike sorting curation.

The GUI can be launched either from the command line:
```bash
uv run sigui --curation --verbose $DATA_FOLDER/full_analyzer_lupin_000939.zarr
```

Or using the `Python` API:

In [ ]:
from spikeinterface_gui import run_mainwindow

In [ ]:
run_mainwindow(analyzer, curation=True)

In [ ]:
with open(results_folder / "manual_curation.json") as f:
    curation_dict = json.load(f)

analyzer_curated = sc.apply_curation(analyzer, curation_dict)

In [ ]:
analyzer_curated

In [ ]:
analyzer_curated.get_sorting_property_keys()

In [ ]:
print("Manual labels:", analyzer_curated.get_sorting_property("quality"))
print("# merged units:", sum(analyzer_curated.get_sorting_property("is_merged")))
print("# split units:", sum(analyzer_curated.get_sorting_property("is_split")))

In [ ]:
# analyzer_curated.save_as(folder="curated_analyzer")

## Automatic curation tools

### Threshold-based curation

This strategy for curation uses different thresholds on quality metrics. Quality metrics are designed to quantify different aspects of a unit, such as amplitude, SNR, drift, ISI violations, spike train completeness, and more.

A ful list of available quality metrics is available [here](https://spikeinterface.readthedocs.io/en/latest/modules/qualitymetrics.html).

In [ ]:
qm = analyzer.get_extension("quality_metrics").get_data()
print(f"Available metrics: {qm.columns.values.to_numpy()}")

`qm` is a `pandas.DataFrame` with unit ID as indices. We can easily perform a query with a quality-metric "recipe", such as:

In [ ]:
sc.threshold_metrics_label_units?

In [ ]:
qc_thresholds = {
    "rp_contamination": {"less": 0.5},
    "presence_ratio": {"greater": 0.8},
    "amplitude_cutoff": {"less": 0.1}
}

In [ ]:
threshold_df = sc.threshold_metrics_label_units(qm, thresholds=qc_thresholds, column_name="threshold_label")
threshold_df

In [ ]:
threshold_df["threshold_label"].value_counts()

Let's see how they look! The `SortingAnalyzer` has a convenient function to select units and propagate all extensions:

In [ ]:
w = sw.plot_unit_labels(analyzer, threshold_df["threshold_label"])

**NICE!!** Most of noisy units are gone, but there are also potential good units in the noise group... maybe our curation was too strict?

### Automatic labeling

In the `curation` module we also provide some model-based automatic labeling of units. 

There are two auti-labeling tools:

* **UnitRefine**: uses pre-trained classifiers from hand-curated data from the [UnitRefine](https://www.biorxiv.org/content/10.1101/2025.03.30.645770v1) project. UniteRefine models use quality and template metrics to classify units as being `noise` or `neural` and `SUA` (single-unit activity) or `MUA` (multi-unit activity). All available models are publicly available on the [SpikeInterface HuggingFace Organization](https://huggingface.co/SpikeInterface).

* **BombCell**: uses a set of advanced heuristics based on quality and template metrics. For more information, see the [Bombcell repo](https://github.com/Julie-Fabre/bombcell)

Let's see how to use them!

In [ ]:
unitrefine_df = sc.unitrefine_label_units(
    analyzer,
    noise_neural_classifier="SpikeInterface/UnitRefine_noise_neural_classifier_lightweight",
    sua_mua_classifier="SpikeInterface/UnitRefine_sua_mua_classifier_lightweight",
)

In [ ]:
unitrefine_df["unitrefine_label"].value_counts()

In [ ]:
w = sw.plot_unit_labels(analyzer, unitrefine_df["unitrefine_label"])

In [ ]:
sc.bombcell_label_units?

In [ ]:
bombcell_thresholds = sc.bombcell_get_default_thresholds()
bombcell_thresholds

In [ ]:
bombcell_df = sc.bombcell_label_units(
    analyzer,
    thresholds=bombcell_thresholds
)

In [ ]:
bombcell_df["bombcell_label"].value_counts()

In [ ]:
w = sw.plot_unit_labels(analyzer, bombcell_df["bombcell_label"])

In [ ]:
threshold_df["threshold_label"].to_numpy(dtype=str)

In [ ]:
# it is also convenient to store the labels in the sorting analyzer, so we can show them later in the GUI
analyzer.set_sorting_property("threshold_label", threshold_df["threshold_label"].to_numpy(dtype=str))
analyzer.set_sorting_property("unitrefine_label", unitrefine_df["unitrefine_label"].to_numpy(dtype=str))
analyzer.set_sorting_property("bombcell_label", bombcell_df["bombcell_label"].to_numpy(dtype=str))

### Auto-merging

Both quality metric-based and model-based curation provide labels for individual units. A common spike sorting mistake, however, is to *oversplit* units, i.e., multiple sorted units are actually from the same underlying neuron.

These errors are usually manually fixed with curation, but the curation module also provides some automated strategies to find potential merges. These strategies have different flavors, but usually use the similarity across templates and auto/cross correlogram shapes to make a merge decision.

See [compute_merge_unit_groups](https://spikeinterface.readthedocs.io/en/latest/api.html#spikeinterface.curation.compute_merge_unit_groups) function for documentation.

In this case, we are going to use the [SLAy](https://www.biorxiv.org/content/10.1101/2025.06.20.660590v3) method.

In [ ]:
potential_merges = sc.compute_merge_unit_groups(
    analyzer,
    preset="slay",
    steps_params={"template_similarity": {"template_diff_thresh": 0.5}}
)

In [ ]:
print(f"Found {len(potential_merges)} potential merges")

### Curation format

We have seen that we can perform several curation actions: labeling units, removing units (usually noisy ones), and merge oversplits.

However, we haven't seen how represent and apply curation steps.
The `curation` module, for this purpose, defines a *curation format*, which is a standardized representation of the curation actions.

In this section we'll see how to formalize a curation, apply to an existing analyzer, and save it to a file that can be used by the [SpikeInterface-GUI](https://github.com/SpikeInterface/spikeinterface-gui), a powerful GUI for visualization and further curation refinement.

In [ ]:
# we will use unitrefine labels as pre-computed "quality"
unitrefine_labels = analyzer.get_sorting_property("unitrefine_label").copy()
unitrefine_labels[unitrefine_labels == "sua"] = "good"
unitrefine_labels[unitrefine_labels == "mua"] = "MUA"

label_definitions = {
    "quality": dict(name="quality", label_options=["good", "MUA", "noise"], exclusive=True),
}
manual_labels = [
    {"unit_id": unit_id, "labels": {"quality": [unitrefine_labels[unit_index]]}}
    for unit_index, unit_id in enumerate(analyzer.unit_ids)
]

In [ ]:
curation_dict = dict(
    format_version="2",
    unit_ids=analyzer.unit_ids,
    label_definitions=label_definitions, 
    merges=[
        dict(unit_ids=p)
        for p in potential_merges
    ],
    removed=analyzer.unit_ids[unitrefine_labels == "noise"],
    manual_labels=manual_labels
)

Now that we have a curation with all steps (remove, merge, label) we can apply it in one go!

In [ ]:
from spikeinterface.curation import validate_curation_dict

In [ ]:
validate_curation_dict(curation_dict)

In [ ]:
# we can use the Curation for more secure serialization
from spikeinterface.curation.curation_model import Curation

curation = Curation(**curation_dict)

In [ ]:
curation.model_dump().keys()

### Curation "refinement" in the SpikeInterface-GUI

Instead of the SpikeInterface-GUI for fully manual curation, we can pre-load it with the automated curation outputs to speed up the curation process. 
The GUI can be now used to review and refine the automated curation.

In [ ]:
run_mainwindow(
    analyzer,
    displayed_unit_properties=["threshold_label", "unitrefine_label", "bombcell_label"],
    curation=True,
    curation_dict=curation.model_dump(),
    mode="desktop"
)

In [ ]:
with open(results_folder / "refined_curation.json") as f:
    refined_curation_dict = json.load(f)

In [ ]:
analyzer_curated = sc.apply_curation(analyzer, refined_curation_dict, sparsity_overlap=0.5)
analyzer_curated